In [1]:
import sys
# TO CHANGE
BASEDIR = "../"
sys.path.insert(0, BASEDIR)

In [2]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.neo4j_functions import Neo4jConnectionConfig
from src.embedding_functions import EmbedderModelConfig, EmbeddingsDatabaseConnectionConfig, VectorDBConnectionConfig

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig
from src.qa_pipeline.knowledge_retriever.cache import KeyValueStoreConfig
from src.qa_pipeline.knowledge_retriever.cache.configs import DEFAULT_KVDB_CONFIGS

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.utils import Logger

/home/dzigen/Desktop/PersonalAI/Personal-AI/pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
main_config = RemoteKnowledgeGraphConfig(
    graph_db_config=Neo4jConnectionConfig(uri='bolt://localhost:7687', user='neo4j', pwd='password', db_name='testing6'),
    embedder_db_config=EmbeddingsDatabaseConnectionConfig(
        nodes_db_config=VectorDBConnectionConfig(
            path='../data/graph_structures/vectorized_nodes/testing6', db_name='vectorized_nodes', is_exist=True, need_to_clear=False
        ),
        triplets_db_config=VectorDBConnectionConfig(
            path='../data/graph_structures/vectorized_triplets/testing6', db_name='vectorized_triplets', is_exist=True, need_to_clear=False
        ),
        embedder_config=EmbedderModelConfig(
            model_name_or_path='../models/intfloat/multilingual-e5-small'
        )
    ),
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method='bfs',
            retriever_config=BFSSearchConfig(),
            cache_config=KeyValueStoreConfig(
                db_vendor='inmemory',
                db_config=DEFAULT_KVDB_CONFIGS['inmemory']
            )
        )
    ),
    log=Logger('main_debug')
)

In [4]:
rkg_main = RemoteKnowledgeGraph(main_config)

No sentence-transformers model found with name ../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [5]:
rkg_main.update_memory(["Every hunter wants to know: where the pheasant sits"])

  0%|          | 0/9 [00:00<?, ?it/s]Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownLabelWarning} {category: UNRECOGNIZED} {title: The provided label is not in the database.} {description: One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: object)} {position: line: 1, column: 13, offset: 12} for query: 'MATCH (subj:object) WHERE subj.name = "hunter" RETURN elementID(subj) as node_id'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your 

all/unique_triplets - 9/9
all/unique_nodes - 18/6


  0%|          | 0/1 [00:00<?, ?it/s]

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


100%|██████████| 1/1 [00:02<00:00,  2.96s/it]

all/unique_triplets - 9/3
all/unique_nodes - 18/6


In [9]:
rkg_main.kg_model.embeddings_db.vectordbs['triplets'].collection.count()

9

In [6]:
rkg_main.update_memory([
    "Mikhail Menshchikov is currently a second-year master's student at ITMO.",
    "Mikhail Menshchikov is studying in the Master's program 'Deep Learning and Generative AI'"])

100%|██████████| 5/5 [00:00<00:00,  9.72it/s]


all/unique_triplets - 5/5
all/unique_nodes - 10/4


  0%|          | 0/1 [00:00<?, ?it/s]

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


100%|██████████| 1/1 [00:01<00:00,  1.26s/it]


...

all/unique_triplets - 8/8
all/unique_nodes - 16/5


  0%|          | 0/1 [00:00<?, ?it/s]Add of existing embedding ID: 4:75034534-a617-4f69-b21f-23a403e8d9a9:6
Insert of existing embedding ID: 4:75034534-a617-4f69-b21f-23a403e8d9a9:6


<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


100%|██████████| 1/1 [00:00<00:00,  3.90it/s]

all/unique_triplets - 8/3
all/unique_nodes - 16/6


In [5]:
rkg_main.answer_question("What program is Mikhail Menshchikov studying for his master's degree?")

Add of existing embedding ID: 4:75034534-a617-4f69-b21f-23a403e8d9a9:6
Number of requested results 20 is greater than number of elements in index 15, updating n_results = 15
Number of requested results 20 is greater than number of elements in index 15, updating n_results = 15
Number of requested results 20 is greater than number of elements in index 15, updating n_results = 15
Number of requested results 50 is greater than number of elements in index 9, updating n_results = 9


<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
extract_thesis, thres 7
<class 'numpy.ndarray'>
['d8b4d611a33eb493ef09b4da187d6893', '69700ddb97181c48cdfc3f5cb3730be8', '208f7b79f68f594eb4b9a9ced69b4b30', '40a7d65e91080f44bd59aed9438a4475']
VectorDBInstance(id=None, document=None, embedding=array([ 4.76047099e-02,  1.24417907e-02, -7.33931586e-02, -2.63290014e-03,
        3.60771231e-02, -5.62042817e-02,  6.57914579e-02,  5.79447336e-02,
        5.81750721e-02,  2.71399114e-02,  4.23642062e-02,  3.16486843e-02,
        5.41700646e-02, -6.17157295e-02, -5.19974008e-02,  2.23023389e-02,
        4.51714545e-02, -1.58476494e-02, -1.06874481e-01, -2.44865809e-02,
        1.47766026e-03, -7.66466791e-03, -5.57579957e-02,  2.31193677e-02,
        3.87971625e-02,  2.15227772e-02, -1.30538410e-02, -5.91033585e-02,
        5.35594765e-04, -7.81652704e-03, -4.90441360e-02, -5.51462024e-02,
        8.04203972e-02, -5.34089729e-02,  8.26904252e-02,  9.89791229e-02,
       -8

'Deep Learning and Generative AI'